# Run plots

Post-run figures for one training run. Set `RUN_DIR`, then run top to bottom.

Per-epoch plots use `EPOCHS`; cross-epoch plots always use every saved epoch.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

from sage.core.logger import setup_logging
from sage.plotting import (
    ValidationPlotManager, best_validated_epoch,
    plot_loss_curves, plot_roc_curve, plot_prediction_raw,
    plot_prediction_probability, plot_calibration_curve, plot_joint_cdfs,
    plot_separation_over_epochs, plot_output_trajectory_over_epochs,
    plot_efficiency_curves, plot_learning_parameter_prior,
    plot_paramfrac_detected_above_thresh, plot_output_vs_param_heatmap,
    plot_correlation_matrix, plot_cumulative_volume,
    plot_2d_efficiency, plot_2d_param_density, plot_confidence_vs_snr,
    plot_output_gradient, plot_diagonal_compare, plot_param_recovery_heatmap,
    plot_pp_calibration,
)
from sage.plotting._epochs import epoch_number

setup_logging()   # so "skipped" messages are visible

# ---- set these ---------------------------------------------------------
RUN_DIR = "/work/nagarajan/sage_runs/o3b/run_export_HL"
EPOCHS  = None       # None = best epoch. Or 45, or [0, 45, "best"], or "all"
# ------------------------------------------------------------------------

plotter = ValidationPlotManager(
    f"{RUN_DIR}/validation_data.h5", f"{RUN_DIR}/losses.h5", export_dir=None,
)

all_epochs = sorted(plotter.validation_data)
selected   = plotter.resolve_epochs(EPOCHS)
ep         = selected[-1]
d          = plotter.validation_data[ep]
sp         = d["source_params"]
best       = best_validated_epoch(plotter.validation_loss)

print(f"{len(all_epochs)} saved epochs | showing {[epoch_number(e) for e in selected]}")
print(f"best epoch {best} (val loss {plotter.validation_loss[best,0]:.4f})")
print(f"point estimates: {d['pe_names']}")

## Loss curves

Training vs validation. Validation is drawn only where it ran.

In [ ]:
plot_loss_curves(plotter.training_loss, plotter.validation_loss, save=False,
                 best_epoch=best, component_names=["total", "bce"] + list(d["pe_names"]))

## ROC

Detection performance.

In [ ]:
plot_roc_curve(ep, d["ranking_stat"], d["labels"], save=False)

## Output distributions

Ranking statistic, raw and after the sigmoid.

In [ ]:
plot_prediction_raw(ep, d["ranking_stat"], d["labels"], save=False)
plot_prediction_probability(ep, d["pred_prob"], d["labels"], save=False)

## Calibration

Reliability of the detection probability, and P-P of the point estimates.

In [ ]:
plot_calibration_curve(ep, d["ranking_stat"], d["labels"], save=False)

if d["pe_names"]:
    names = list(d["pe_names"])
    plot_pp_calibration(
        mu   =np.column_stack([d["pe_pred"][p]  for p in names]),
        sigma=np.column_stack([d["pe_sigma"][p] for p in names]),
        y    =np.column_stack([d["pe_true"][p]  for p in names]),
        param_names=names, epoch=ep, save=False,
    )

## Joint CDFs

Cumulative signal and noise distributions.

In [ ]:
plot_joint_cdfs(ep, d["ranking_stat"], d["labels"], save=False)

## Separation over epochs

Signal/noise pulling apart. Colour is epoch. Uses all epochs.

In [ ]:
plot_separation_over_epochs(
    {e: plotter.validation_data[e]["ranking_stat"] for e in all_epochs},
    {e: plotter.validation_data[e]["labels"] for e in all_epochs},
    all_epochs, save=False)

## Output distribution over epochs

Median and percentile bands per epoch. Uses all epochs.

In [ ]:
plot_output_trajectory_over_epochs(
    [plotter.validation_data[e]["ranking_stat"] for e in all_epochs],
    [plotter.validation_data[e]["labels"] for e in all_epochs],
    [epoch_number(e) for e in all_epochs], save=False)

## Efficiency vs parameter

Ranking statistic against each source parameter.

In [ ]:
plot_efficiency_curves(ep, sp, d["ranking_stat"], d["labels"], save=False)

## Learning the prior

Whether the network absorbed the prior rather than the physics.

In [ ]:
plot_learning_parameter_prior(ep, sp, d["ranking_stat"], d["labels"], save=False)

## Detected fraction

Recovered fraction per parameter bin.

In [ ]:
plot_paramfrac_detected_above_thresh(ep, d["ranking_stat"], d["labels"], sp, save=False)

## Output vs parameter

Ranking-statistic density against each parameter.

In [ ]:
for p in ("mchirp", "distance", "q"):
    if p in sp:
        plot_output_vs_param_heatmap(ep, d["ranking_stat"], d["labels"], sp, p, save=False)

## Output gradient

d(ranking statistic)/d(parameter), from the binned median trend.

In [ ]:
for p in ("mchirp", "distance"):
    if p in sp:
        plot_output_gradient(ep, d["ranking_stat"], d["labels"], sp, p, save=False)

## Correlations

Only parameters the ranking statistic actually correlates with.

In [ ]:
plot_correlation_matrix(
    d["ranking_stat"],
    plotter._correlated_params(d["ranking_stat"], sp, d["labels"]),
    d["labels"], epoch=ep, save=False)

## Cumulative volume

Sensitive volume vs distance.

In [ ]:
plot_cumulative_volume(ep, d["ranking_stat"], d["labels"], sp, "distance", save=False)

## Two-parameter views

Efficiency and density over parameter pairs.

In [ ]:
for px, py in plotter.param_pairs:
    if px in sp and py in sp:
        plot_2d_efficiency(ep, d["ranking_stat"], d["labels"], sp, px, py, 0.5, save=False)
        plot_2d_param_density(ep, d["ranking_stat"], d["labels"], sp, px, py, save=False)

## Confidence vs SNR

Needs a recorded network SNR; skipped otherwise (chirp distance is not a substitute).

In [ ]:
snr_key = next((k for k in ("snr", "network_snr") if k in sp), None)
if snr_key:
    plot_confidence_vs_snr(ep, d["ranking_stat"], d["labels"], sp[snr_key], save=False)
else:
    print("no network SNR recorded in this run - skipped")

## Point-estimate recovery

Predicted vs true, physical units on both axes.

In [ ]:
if d["pe_names"]:
    plot_diagonal_compare(
        ep, d["pe_pred"], d["pe_true"], None,
        np.ones(len(next(iter(d["pe_pred"].values())))), save=False)

## Recovery across epochs

Residual per parameter bin over all epochs.

In [ ]:
for p in d["pe_names"]:
    plot_param_recovery_heatmap(
        {e: plotter.validation_data[e]["pe_pred"] for e in all_epochs},
        {e: plotter.validation_data[e]["pe_true"] for e in all_epochs},
        p, all_epochs, save=False)

## Write everything to disk

Full suite into the run directory.

In [ ]:
# plotter.export_dir = RUN_DIR
# plotter.make_all_plots(save=True, epochs=EPOCHS)